# 01b. Multicontact extracellular stimulation and recording

In the previous tutorial, we had a single extracellular stimulating electrode positioned near a neuron. In this tutorial, we will explore how to set up more complex extracellular stimulation and recording scenarios using multiple electrodes.

As before. we will simulate a single myelinated axon. However, now the model will be receiving extracellular kHz stimulation from 2 point source electrodes, while recording the extracellular potential at 2 other nearby points.

In [ ]:
import axonml as ax
from axonml.models.mod import hh
from axonml.units import mm, um

model = ax.Myelinated(
    diameters=[6.0], n_node=201, node_length=1.0, celsius=6.3, v_init=-65.0
)
model.insert(hh)

AxonML makes it easy to define multiple extracellular electrodes and configure their positions, stimulation parameters, and recording settings. This allows for more realistic simulations of neural responses to complex stimulation patterns, as well as the ability to capture extracellular potentials at multiple locations around the neuron. The only diifference in code from the previous tutorial is defining multiple electrodes and setting up their parameters accordingly.

In [ ]:
s1 = ax.isotropic_point(x=-40 * mm, y=0.0 * mm, z=200.0 * um)
s2 = ax.isotropic_point(x=40 * mm, y=0.0 * mm, z=-200.0 * um)

Lets first model the effect of each individually:

In [ ]:
from axonml.units import Hz

rec = ax.callbacks.Recorder(['v'])
stim1 = ax.sin(amp=0.1, freq=100*Hz)

tstop, dt = 50.0, 0.025

rec.reset()
model.initialize()
model.longrun(
    tstop=tstop,
    dt=dt,
    chunklength=100,
    extra=(s1(model), stim1),
    callbacks=[rec],
)

In [ ]:
#some plotting utilities to make movies

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

def movie_profile(
    y_tx,
    x,
    out,
    *,
    fps=30,
    t=None,
    xlabel="x",
    ylabel="value",
    title=None,
    ylim=None,
    xlim=None,
    sort_x=True,
    dpi=150,
    bitrate=2000,
    codec="h264",
    writer="ffmpeg",
    extra_ffmpeg_args=("-pix_fmt", "yuv420p"),
):
    """
    Save an MP4 (or GIF) movie of a 1D profile y(x) evolving over time.

    Parameters
    ----------
    y_tx : array-like, shape (n_timesteps, n_locations)
        Values over time (rows) and space (columns).
    x : array-like, shape (n_locations,)
        Spatial coordinates corresponding to columns of y_tx.
    out : str or pathlib.Path
        Output filename, e.g. "profile.mp4" or "profile.gif".
    fps : int
        Frames per second.
    t : array-like, optional
        Time values for each frame (n_timesteps,). If provided, used in the title.
    xlabel, ylabel : str
        Axis labels.
    title : str, optional
        Base title. If None, only the time/frame is shown.
    ylim, xlim : tuple, optional
        Axis limits. If None, computed from data.
    sort_x : bool
        If True, sort x (and corresponding columns) for a clean line plot.
    dpi : int
        Output DPI.
    bitrate : int
        Video bitrate (kbps-ish; matplotlib passes through to ffmpeg).
    codec : str
        ffmpeg codec, default "h264".
    writer : {"ffmpeg","pillow"}
        Writer backend. Use "ffmpeg" for mp4, "pillow" for gif.
    extra_ffmpeg_args : tuple[str, ...]
        Extra arguments passed to ffmpeg (helps browser compatibility).

    Returns
    -------
    str
        The output path as a string.
    """
    y_tx = np.asarray(y_tx)
    x = np.asarray(x)

    if y_tx.ndim != 2:
        raise ValueError(f"y_tx must be 2D (n_timesteps, n_locations); got shape {y_tx.shape}")
    if x.ndim != 1 or x.shape[0] != y_tx.shape[1]:
        raise ValueError(f"x must be 1D with length n_locations={y_tx.shape[1]}; got shape {x.shape}")

    if t is not None:
        t = np.asarray(t)
        if t.shape[0] != y_tx.shape[0]:
            raise ValueError(f"t must have length n_timesteps={y_tx.shape[0]}; got shape {t.shape}")

    if sort_x:
        order = np.argsort(x)
        x = x[order]
        y_tx = y_tx[:, order]

    # Robust limits (ignore NaNs)
    if xlim is None:
        xlim = (np.nanmin(x), np.nanmax(x))
    if ylim is None:
        y0 = np.nanmin(y_tx)
        y1 = np.nanmax(y_tx)
        pad = 0.05 * (y1 - y0 if y1 > y0 else 1.0)
        ylim = (y0 - pad, y1 + pad)

    fig, ax = plt.subplots()
    (line,) = ax.plot(x, y_tx[0], lw=2)

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)

    title_text = ax.set_title("")

    interval_ms = 1000 / fps

    def _frame_title(i):
        base = "" if title is None else str(title)
        if t is None:
            time_part = f"frame {i}"
        else:
            time_part = f"t = {t[i]:.6g}"
        return f"{base} ({time_part})" if base else time_part

    def update(i):
        line.set_ydata(y_tx[i])
        title_text.set_text(_frame_title(i))
        return line, title_text

    ani = FuncAnimation(fig, update, frames=y_tx.shape[0], interval=interval_ms, blit=True)

    out = str(out)
    ext = out.lower().rsplit(".", 1)[-1]

    if writer == "ffmpeg" or ext == "mp4":
        try:
            from matplotlib.animation import FFMpegWriter
        except Exception as e:
            plt.close(fig)
            raise RuntimeError(
                "FFMpegWriter unavailable. Install ffmpeg (and ensure it's on PATH), "
                "or set writer='pillow' to save a GIF."
            ) from e

        w = FFMpegWriter(
            fps=fps,
            codec=codec,
            bitrate=bitrate,
            extra_args=list(extra_ffmpeg_args) if extra_ffmpeg_args else None,
        )
        ani.save(out, writer=w, dpi=dpi)
    else:
        # GIF via Pillow
        try:
            from matplotlib.animation import PillowWriter
        except Exception as e:
            plt.close(fig)
            raise RuntimeError("PillowWriter unavailable. Install pillow, or use writer='ffmpeg' for mp4.") from e

        w = PillowWriter(fps=fps)
        ani.save(out, writer=w, dpi=dpi)

    plt.close(fig)
    return out


In [ ]:
from pathlib import Path

out = Path("../_static/media/s1.mp4")
out.parent.mkdir(parents=True, exist_ok=True)

movie_profile(rec.numpy('v')[:, 0, :], model.x.flatten().numpy(), out, fps=30, title="s1")

In [ ]:
from IPython.display import Video, display
display(Video(str(out), embed=True))

In [ ]:
stim2 = ax.sin(amp=0.1, freq=200*Hz, delay=5.0)

tstop, dt = 50.0, 0.025

rec.reset()
model.initialize()
model.longrun(
    tstop=tstop,
    dt=dt,
    chunklength=100,
    extra=(s2(model), stim2),
    callbacks=[rec],
)

out = Path("../_static/media/s2.mp4")
out.parent.mkdir(parents=True, exist_ok=True)

movie_profile(rec.numpy('v')[:, 0, :], model.x.flatten().numpy(), out, fps=30, title="s2")
display(Video(str(out), embed=True))


Applying both simultaneously is as simple as adding both to the `extra` argument of the `longrun` function.

In [ ]:
rec.reset()
model.initialize()
model.longrun(
    tstop=tstop,
    dt=dt,
    chunklength=100,
    extra=[(s1(model), stim1), (s2(model), stim2)],
    callbacks=[rec],
)

out = Path("../_static/media/s1_and_s2.mp4")
out.parent.mkdir(parents=True, exist_ok=True)

movie_profile(rec.numpy('v')[:, 0, :], model.x.flatten().numpy(), out, fps=30, title="s1_and_s2")
display(Video(str(out), embed=True))


AxonML will handle the superposition of the extracellular potentials from both electrodes automatically, allowing us to simulate the combined effect of multiple stimulation sources on the neuron. You can supply as many electrodes as needed in the `extra` argument, making it straightforward to set up complex extracellular stimulation scenarios.